# UTTA-Med — Seed 99 (frozen recipe from seed 42)

GPU T4, Internet ON. One session: source train → Tent/EATA → UTTA-Med.

**Frozen (do not retune):**
- BN running stats frozen, Adam TTA **lr = 1e-5**
- N_MC = 20
- τ = **70th percentile** of this seed's OOD-val MC-Dropout U (not seed-42's absolute τ)
- EATA e_margin = 0.4
- Matched coverage on val for random / confidence / entropy
- Test hospital 2 **once** after τ is frozen


In [ ]:
!pip -q install datasets pyarrow scikit-learn tqdm pillow
import torch
assert torch.cuda.is_available(), 'Enable GPU T4'
print('GPU', torch.cuda.get_device_name(0))

In [ ]:
import json, random
from pathlib import Path
import numpy as np, torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm
from datasets import load_dataset

SEED, SRC_LR, TTA_LR, N_MC = 99, 1e-4, 1e-5, 20
EPOCHS, PATIENCE, BS_TRAIN, BS_EVAL = 20, 5, 128, 64
E_MARGIN = 0.4
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda')
print('SEED', SEED)

In [ ]:
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
train_tf = transforms.Compose([
    transforms.Resize((96,96)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10), transforms.ColorJitter(0.1,0.1,0.1,0.02),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
eval_tf = transforms.Compose([transforms.Resize((96,96)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

def cid(v):
    return int(v.item() if hasattr(v,'item') else v)

class HFCamelyon(Dataset):
    def __init__(self, split, centers, tf, with_meta=False):
        centers={int(c) for c in centers}
        keep=[i for i,c in enumerate(split['center']) if cid(c) in centers]
        self.ds = split.select(keep) if len(keep)<len(split) else split
        self.tf, self.with_meta = tf, with_meta
        print(' kept', len(self.ds), sorted(centers))
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        r=self.ds[i]; img=r['image']
        img = img.convert('RGB') if isinstance(img, Image.Image) else Image.fromarray(np.array(img)).convert('RGB')
        x, y = self.tf(img), torch.tensor(float(int(r['label'])))
        if not self.with_meta:
            return x, y
        slide = r.get('slide', r.get('patient', -1))
        try: slide=int(slide)
        except Exception: slide=hash(str(slide)) % 10_000_000
        return x, y, slide

class Net(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        w = models.ResNet18_Weights.DEFAULT if pretrained else None
        bb=models.resnet18(weights=w); d=bb.fc.in_features; bb.fc=nn.Identity(); self.backbone=bb
        self.classifier=nn.Sequential(nn.Linear(d,d), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(d,1))
    def forward(self,x): return self.classifier(self.backbone(x))

raw=load_dataset('wltjr1007/Camelyon17-WILDS')
train_set=HFCamelyon(raw['train'],{0,3,4}, train_tf)
id_set=HFCamelyon(raw['validation'],{0,3,4}, eval_tf)
ood_set=HFCamelyon(raw['validation'],{1}, eval_tf)
test_set=HFCamelyon(raw['test'],{2}, eval_tf, with_meta=True)
train_loader=DataLoader(train_set,batch_size=BS_TRAIN,shuffle=True,num_workers=0,pin_memory=True)
id_loader=DataLoader(id_set,batch_size=BS_EVAL,shuffle=False,num_workers=0,pin_memory=True)
ood_loader=DataLoader(ood_set,batch_size=BS_EVAL,shuffle=False,num_workers=0,pin_memory=True)
test_loader=DataLoader(test_set,batch_size=BS_EVAL,shuffle=False,num_workers=0,pin_memory=True)

In [ ]:
def ece_score(y,p,n=15):
    y,p=np.asarray(y).ravel(),np.asarray(p).ravel(); bins=np.linspace(0,1,n+1); e=0.0
    for i in range(n):
        m=(p>bins[i])&(p<=bins[i+1]) if i else (p>=bins[i])&(p<=bins[i+1])
        if m.any(): e += m.mean()*abs(y[m].mean()-p[m].mean())
    return float(e)
def metrics_of(y,p):
    y,p=np.asarray(y).ravel(),np.asarray(p).ravel(); pred=(p>=0.5).astype(int)
    return dict(n=int(len(y)), accuracy=float(accuracy_score(y,pred)),
        precision=float(precision_score(y,pred,zero_division=0)), recall=float(recall_score(y,pred,zero_division=0)),
        sensitivity=float(recall_score(y,pred,zero_division=0)),
        specificity=float(((y==0)&(pred==0)).sum()/max((y==0).sum(),1)),
        f1=float(f1_score(y,pred,zero_division=0)), auroc=float(roc_auc_score(y,p)),
        auprc=float(average_precision_score(y,p)), ece=ece_score(y,p), brier=float(np.mean((p-y)**2)))
def flip_table(y,b,a):
    y=np.asarray(y).ravel().astype(int); b=(np.asarray(b).ravel()>=0.5).astype(int); a=(np.asarray(a).ravel()>=0.5).astype(int)
    return dict(correction_rate=float(((b!=y)&(a==y)).sum()/max((b!=y).sum(),1)),
                harm_rate=float(((b==y)&(a!=y)).sum()/max((b==y).sum(),1)), flip_rate=float((a!=b).mean()),
                correction=int(((b!=y)&(a==y)).sum()), harmful=int(((b==y)&(a!=y)).sum()))
def bn_affine(m):
    for p in m.parameters(): p.requires_grad=False
    ps=[]
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d) and mod.affine:
            mod.weight.requires_grad=True; mod.bias.requires_grad=True; ps += [mod.weight, mod.bias]
    return ps
def eval_mode(m):
    m.eval()
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d): mod.eval()
def mc_mode(m):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d): mod.eval()
        elif isinstance(mod, nn.Dropout): mod.train()
def tta_mode(m):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.Dropout): mod.eval()
        elif isinstance(mod, nn.BatchNorm2d): mod.eval()
def entropy(z):
    p=torch.sigmoid(z).clamp(1e-6,1-1e-6)
    return -(p*torch.log(p)+(1-p)*torch.log(1-p)).view(-1)
@torch.no_grad()
def mc_predict(m, x, n=N_MC):
    mc_mode(m); ps=[torch.sigmoid(m(x)).view(-1) for _ in range(n)]
    P=torch.stack(ps,0); return P.mean(0), P.var(0, unbiased=False)
@torch.no_grad()
def evaluate(m, ld, has_meta=False):
    eval_mode(m); ys,ps=[],[]
    for batch in ld:
        x,y = batch[0], batch[1]
        x=x.to(device); ps.append(torch.sigmoid(m(x)).view(-1).cpu().numpy()); ys.append(y.view(-1).numpy())
    return metrics_of(np.concatenate(ys), np.concatenate(ps))

In [ ]:
CKPT = WORK / f'camelyon17_resnet18_source_s{SEED}_BEST.pt'
# reuse ckpt if this session already trained
if CKPT.exists():
    st=torch.load(CKPT, map_location='cpu', weights_only=False)
    print('loaded existing', CKPT, 'epoch', st.get('epoch'), 'val_auroc', st.get('val_auroc'))
else:
    print('Training source seed', SEED, 'save-best-only, early-stop on hospital 1')
    model=Net(pretrained=True).to(device)
    opt=torch.optim.Adam(model.parameters(), lr=SRC_LR, weight_decay=1e-4)
    crit=nn.BCEWithLogitsLoss(); scaler=torch.amp.GradScaler('cuda')
    best,stale,best_state,best_ep=-1.0,0,None,-1
    history=[]
    for epoch in range(1, EPOCHS+1):
        model.train(); run=n=0
        for x,y in tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}'):
            x,y=x.to(device), y.view(-1,1).to(device); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'): loss=crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run+=loss.item()*x.size(0); n+=x.size(0)
        val=evaluate(model, ood_loader)
        history.append({'epoch':epoch,'train_loss':run/max(n,1), **{f'val_{k}':v for k,v in val.items()}})
        print(f"epoch {epoch:02d} loss={run/max(n,1):.4f} val_auroc={val['auroc']:.4f} val_f1={val['f1']:.4f}")
        if val['auroc']>best:
            best,stale,best_ep=val['auroc'],0,epoch
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            torch.save({'model':best_state,'epoch':epoch,'val_auroc':best,'seed':SEED,'history':history}, CKPT)
            print('  saved BEST', CKPT)
        else:
            stale += 1
            if stale>=PATIENCE:
                print('early stop at', epoch, 'best', best); break
    model.load_state_dict(best_state)
    st=torch.load(CKPT, map_location='cpu', weights_only=False)

def load_fresh():
    m=Net(pretrained=False).to(device); m.load_state_dict(st['model']); return m

src=load_fresh()
ood_src=evaluate(src, ood_loader)
id_src=evaluate(src, id_loader)
print('ASSERT OOD', round(ood_src['auroc'],4), 'ckpt', st.get('val_auroc'))
assert abs(ood_src['auroc']-float(st['val_auroc']))<0.01, 'restore mismatch'

In [ ]:
print('--- MC-Dropout U vs error (hospital 1) ---')
src=load_fresh(); ys,Us,Ps,Hs,Cs=[],[],[],[],[]
for x,y in tqdm(ood_loader, desc='mc val'):
    x=x.to(device); mean,var=mc_predict(src,x); p=mean.clamp(1e-6,1-1e-6)
    ys.append(y.view(-1).numpy()); Us.append(var.cpu().numpy()); Ps.append(mean.cpu().numpy())
    Hs.append((-(p*torch.log(p)+(1-p)*torch.log(1-p))).cpu().numpy())
    Cs.append(torch.maximum(p,1-p).cpu().numpy())
y=np.concatenate(ys); U=np.concatenate(Us); P=np.concatenate(Ps); H=np.concatenate(Hs); C=np.concatenate(Cs)
err=((P>=0.5).astype(int)!=y.astype(int)).astype(float)
order=np.argsort(U); u_bins=[]
print('bin n meanU err')
for i,idx in enumerate(np.array_split(order,10)):
    row=dict(bin=i+1,n=int(len(idx)),mean_U=float(U[idx].mean()),err_rate=float(err[idx].mean()))
    u_bins.append(row); print(row['bin'], row['n'], f"{row['mean_U']:.6g}", f"{row['err_rate']:.4f}")
corr=float(np.corrcoef(U,err)[0,1]); print('corr(U,error)', round(corr,4))
tau=float(np.quantile(U, 0.70))
print('FROZEN RULE tau = 70th percentile of val U =', tau)

In [ ]:
def unpack(batch):
    if len(batch)==3: return batch[0], batch[1], batch[2]
    return batch[0], batch[1], None

def adapt(ld, gate, tau=None, target_cov=None, e_margin=E_MARGIN, has_meta=False, rng_seed=SEED):
    m=load_fresh(); srcm=load_fresh(); eval_mode(srcm)
    opt=torch.optim.Adam(bn_affine(m), lr=TTA_LR)
    rng=np.random.RandomState(rng_seed)
    n_seen=n_kept=n_skip=0; ys,pb,pa,slides=[],[],[],[]
    for batch in tqdm(ld, desc=str(gate)):
        x,y,sl = unpack(batch)
        x=x.to(device); B=x.size(0); n_seen += B
        with torch.no_grad():
            eval_mode(srcm); p0=torch.sigmoid(srcm(x)).view(-1); pb.append(p0.cpu().numpy())
            if gate=='tent':
                w=torch.ones(B, device=x.device)
            elif gate=='eata':
                w=(entropy(srcm(x))<e_margin).float()
            elif gate=='random':
                w=torch.tensor(rng.rand(B)<target_cov, device=x.device, dtype=torch.float32)
            elif gate=='confidence':
                w=(torch.maximum(p0,1-p0)>tau).float()
            elif gate=='entropy':
                w=(entropy(torch.logit(p0.clamp(1e-6,1-1e-6)))<tau).float()
            elif gate=='utta':
                _, var = mc_predict(srcm, x); w=(var<tau).float()
            else:
                raise ValueError(gate)
        tta_mode(m); k=int(w.sum().item())
        if k==0:
            n_skip += 1
        else:
            opt.zero_grad(set_to_none=True)
            loss=(w*entropy(m(x))).sum()/w.sum(); loss.backward(); opt.step(); n_kept += k
        with torch.no_grad():
            eval_mode(m); pa.append(torch.sigmoid(m(x)).view(-1).cpu().numpy())
        ys.append(y.view(-1).numpy())
        if sl is not None: slides.append(np.asarray(sl).reshape(-1))
    yv,b,a=np.concatenate(ys),np.concatenate(pb),np.concatenate(pa)
    out=dict(gate=gate, tau=None if tau is None else float(tau), target_cov=target_cov, lr=TTA_LR,
             coverage=n_kept/max(n_seen,1), n_skip_batch=n_skip,
             before=metrics_of(yv,b), after=metrics_of(yv,a), harm=flip_table(yv,b,a))
    if slides:
        out['n_slides']=int(len(np.unique(np.concatenate(slides))))
    return out

target_cov=float((U<tau).mean())
conf_t=float(np.quantile(C, 1.0-target_cov))
ent_t=float(np.quantile(H, target_cov))
print('matched val coverage', round(target_cov,3), 'conf_t', conf_t, 'ent_t', ent_t)

print('--- TEST once (hospital 2) ---')
src_test=evaluate(load_fresh(), test_loader, has_meta=True)
tent=adapt(test_loader,'tent', has_meta=True)
eata=adapt(test_loader,'eata', has_meta=True)
rand=adapt(test_loader,'random', target_cov=target_cov, has_meta=True)
conf=adapt(test_loader,'confidence', tau=conf_t, has_meta=True)
entg=adapt(test_loader,'entropy', tau=ent_t, has_meta=True)
utta=adapt(test_loader,'utta', tau=tau, has_meta=True)

def line(name, r, src=False):
    m=r if src else r['after']
    cov='—' if src else f"{r['coverage']:.3f}"
    harm='—' if src else f"{r['harm']['harm_rate']:.4f}"
    print(f"{name:12s} AUROC={m['auroc']:.4f} F1={m['f1']:.4f} ECE={m['ece']:.4f} cov={cov} harm={harm}")

print('=== SEED', SEED, 'TEST ===')
line('source', src_test, True)
line('tent', tent); line('eata', eata); line('random', rand)
line('confidence', conf); line('entropy', entg); line('utta', utta)

id_m=id_src
out={
    'cuda': True, 'device': torch.cuda.get_device_name(0), 'seed': SEED,
    'checkpoint': str(CKPT), 'best_epoch': int(st.get('epoch',-1)),
    'protocol': {'bn_mode':'bn_freeze_stats','tta_lr':TTA_LR,'n_mc':N_MC,
                 'tau_rule':'70th percentile of val U','e_margin':E_MARGIN},
    'tau': tau, 'matched_coverage_val': target_cov,
    'conf_threshold': conf_t, 'entropy_threshold': ent_t,
    'u_error_corr': corr, 'u_bins': u_bins,
    'id_val': id_m, 'val_ood': ood_src, 'source_test': src_test,
    'shift': {'auroc_drop': id_m['auroc']-src_test['auroc'], 'f1_drop': id_m['f1']-src_test['f1']},
    'tent_test': tent, 'eata_test': eata, 'random_test': rand,
    'confidence_test': conf, 'entropy_test': entg, 'utta_test': utta,
}
p=WORK/f'camelyon17_resnet18_full_s{SEED}.json'
p.write_text(json.dumps(out, indent=2))
print('wrote', p)
print('also download', CKPT)